# Transformacion de Shp Files a Gpkg

Este script se encarga de:

- Leer los archivos .shp provistos por el dane
- Realizar la copia local de la transformacion de fichero GPKG
- Cargar los archivos transformados a S3 para posterior carga a Athena

<a id='contents' />

## Tabla de contenidos

1. [Carga de librerías](#loading)
2. [Definicion de metodos](#methods)
3. [Definicion de variables](#variables)
4. [Scripts principales](#scripts)

<a id='loading' />

## Carga de librerías
[(back to top)](#contents)

In [5]:
%load_ext autoreload
%autoreload 2
%run config_notebook.py

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Directorio "src" adicionado para cargar modulos


ModuleNotFoundError: No module named 'sagemaker'

In [6]:
import geopandas as gpd
from datetime import datetime
import boto3
import os

ModuleNotFoundError: No module named 'geopandas'

<a id='methods' />

## Definicion de metodos
[(back to top)](#contents)

In [ ]:
def set_shapes_bucket(ssm, aws_account_env):
    env_name = "prod" if aws_account_env == "produccion" else "dev"
    if (aws_account_env != "hel001"):
        return f"nutresa-datalake-{env_name}-raw"
    else:
        return ssm.get_parameter(Name='/ml/s3/foundation/feature_store_bucket_name')['Parameter']['Value']
    
def set_monitoring_name(ssm, aws_account_env):
    return ssm.get_parameter(Name='/ml/s3/foundation/monitoring_bucket_name')['Parameter']['Value']
    
def set_dane_folder_name(env_name, is_manzana):
    if (env_name != "hel001"):
        path = "engineering/datos-no-estructurados_tranversales/mapas_evp/ftp/mgn_municipio_politico/"
        if is_manzana:
            path = "engineering/datos-no-estructurados_tranversales/mapas_evp/ftp/manzana/"
    else:
        path = "Dane_shapes/MGN2020_MPIO_POLITICO/"
        if is_manzana:
            path = "Dane_shapes/MANZANA/"
    return path

<a id='variables' />

## Definicion de variables
[(back to top)](#contents)

In [ ]:
ssm = boto3.client('ssm', region_name='us-east-1')
aws_account_env = ssm.get_parameter(Name='/ml/misc/foundation/env_name')['Parameter']['Value']
s3 = boto3.resource('s3')
shapes_bucket = set_shapes_bucket(ssm, aws_account_env)
print(f"shapes_bucket: {shapes_bucket}")
bucket = s3.Bucket(shapes_bucket)
dane_folder = set_dane_folder_name(aws_account_env, True)
path = f"s3://{shapes_bucket}/{dane_folder}"
print(f"path_by_muni: {path}")

<a id='scripts' />

## Scripts principales
[(back to top)](#contents)

Verificacion de registros en Bucket

In [ ]:
shp_files = bucket.objects.filter(Prefix=dane_folder).all()
shp_file_count = 0
for file in shp_files:
    if file.key.endswith('.shp'):
        shp_file_count += 1

print(f"shp_file_count: {shp_file_count}" + "\n")

Verificacion de Bucket de salida

In [ ]:
monitoring_bucket = set_monitoring_name(ssm, aws_account_env)
output_bucket = s3.Bucket(monitoring_bucket)
monitoring_bucket

Carga de archivos geopackages a S3

In [ ]:
%%time
current_file_count = 1
for file in shp_files:
    if file.key.endswith('.shp'):
        shp_file_path = f"s3://{shapes_bucket}/{file.key}"
        shp_file = file.key.replace(dane_folder, "")
        print("="*30 + "\n")
        print(f"shp_file: {shp_file}")
        shape_muni = gpd.read_file(shp_file_path)
        new_shp_file = shp_file.replace(".shp", ".gpkg")
        print(f"new_shp_file: {new_shp_file}")
        gpkg_file_path = f"geoPackages/{new_shp_file}"
        shape_muni.to_file(gpkg_file_path, driver='GPKG')
        
        with open(gpkg_file_path, 'rb') as data:
            output_bucket.put_object(Key=gpkg_file_path, Body=data)
        if os.path.exists(gpkg_file_path):
            os.remove(gpkg_file_path)
        
        print(f"new_shp_file: {new_shp_file} OK {current_file_count}/{shp_file_count}" + "\n")
        print("="*30 + "\n")
        current_file_count += 1